[← Back to 08b - Changing Data, Part 2](<../08b-changing-data-part-2.ipynb>) · [Course Overview](<../00-course-overview.ipynb>)

# 08b - Changing Data, Part 2: Solutions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samietex/SQL_Tutorial/blob/main/solutions/08b-changing-data-part-2-solutions.ipynb)

Try each exercise in the lesson notebook first. If your query differs from the one here but returns the same rows, it may well be just as correct, and seeing why is a useful exercise in itself.

In [1]:
# Run this cell first. It downloads the small helper that lets the cells below run SQL,
# fetching a fresh copy every time so you always have the latest version.
import os, sys, tempfile, urllib.request
helper_dir = os.path.join(tempfile.gettempdir(), 'sql_tutorial_helper')
os.makedirs(helper_dir, exist_ok=True)
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/samietex/SQL_Tutorial/assets/sql_tutorial.py',
    os.path.join(helper_dir, 'sql_tutorial.py'),
)
sys.path.insert(0, helper_dir)
%reload_ext sql_tutorial

Ready. %%sql runs SQLite 3.50.4 in memory, %%tsql runs SQL Server.


In [2]:
%%sql
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT NOT NULL,
    credit_limit INTEGER NOT NULL
);
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    amount INTEGER NOT NULL
);
INSERT INTO customers VALUES
    (1, 'Ada', 100),
    (2, 'Bola', 200),
    (3, 'Chidi', 300),
    (4, 'Dami', 200);
INSERT INTO orders VALUES
    (101, 1, 40),
    (102, 1, 60),
    (103, 2, 80),
    (104, 2, 90),
    (105, 3, 140);

4 rows inserted
5 rows inserted


In [3]:
%%sql
CREATE TABLE payments (
    payment_id INTEGER PRIMARY KEY,
    order_id INTEGER NOT NULL,
    amount INTEGER NOT NULL CHECK (amount > 0),
    method TEXT NOT NULL DEFAULT 'card' CHECK (method IN ('card', 'cash', 'transfer')),
    reference TEXT UNIQUE
);

In [4]:
%%sql
INSERT INTO payments (
    payment_id,
    order_id,
    amount,
    reference
)
VALUES (1, 101, 40, 'PAY-001');

1 row inserted


## 1. Design a table

In [5]:
%%sql
CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT NOT NULL UNIQUE,
    price INTEGER NOT NULL CHECK (price > 0),
    in_stock INTEGER NOT NULL DEFAULT 1
);
INSERT INTO products (
    product_id,
    product_name,
    price
)
VALUES (1, 'Notebook', 12);
SELECT
    product_id,
    product_name,
    price,
    in_stock
FROM products;

1 row inserted


product_id,product_name,price,in_stock
1,Notebook,12,1


`in_stock` came out as 1 without being mentioned. Now one failing insert per rule, each meant to fail: a missing name, a repeated name, and a price of 0.

In [6]:
%%sql --expect-error
INSERT INTO products (
    product_id,
    product_name,
    price
)
VALUES (2, NULL, 5);

Error (this cell is meant to fail): NOT NULL constraint failed: products.product_name


In [7]:
%%sql --expect-error
INSERT INTO products (
    product_id,
    product_name,
    price
)
VALUES (3, 'Notebook', 9);

Error (this cell is meant to fail): UNIQUE constraint failed: products.product_name


In [8]:
%%sql --expect-error
INSERT INTO products (
    product_id,
    product_name,
    price
)
VALUES (4, 'Pen', 0);

Error (this cell is meant to fail): CHECK constraint failed: price > 0


Testing a rule with a row that should fail is the only way to know the rule is really there; a typo in a `CHECK` can make it accept everything.

## 2. Which inserts get in?

(a) succeeds, with `method` defaulting to `card`. (b) fails, because the amount must be greater than 0. (c) fails, because `'PAY-001'` is already used. (d) succeeds, because `UNIQUE` allows another `NULL`. Here they are; (b) and (c) are meant to fail:

In [9]:
%%sql
INSERT INTO payments (
    payment_id,
    order_id,
    amount,
    reference
)
VALUES (7, 103, 80, 'PAY-007');

1 row inserted


In [10]:
%%sql --expect-error
INSERT INTO payments (
    payment_id,
    order_id,
    amount,
    method,
    reference
)
VALUES (8, 104, 0, 'cash', 'PAY-008');

Error (this cell is meant to fail): CHECK constraint failed: amount > 0


In [11]:
%%sql --expect-error
INSERT INTO payments (
    payment_id,
    order_id,
    amount,
    method,
    reference
)
VALUES (9, 105, 140, 'card', 'PAY-001');

Error (this cell is meant to fail): UNIQUE constraint failed: payments.reference


In [12]:
%%sql
INSERT INTO payments (
    payment_id,
    order_id,
    amount,
    method,
    reference
)
VALUES (10, 104, 90, 'transfer', NULL);

1 row inserted


## 3. Add a column safely

In [13]:
%%sql
BEGIN;

ALTER TABLE customers
ADD COLUMN signup_year INTEGER NOT NULL DEFAULT 2026;

SELECT
    customer_id,
    customer_name,
    signup_year
FROM customers;

ROLLBACK;

customer_id,customer_name,signup_year
1,Ada,2026
2,Bola,2026
3,Chidi,2026
4,Dami,2026


Every customer gets 2026, and `NOT NULL` is allowed because the `DEFAULT` gives the existing rows a value. `ALTER TABLE` can be rolled back like any other statement in SQLite, which isn't true everywhere: MySQL commits structural changes immediately, even inside a transaction, so check before relying on it.

## 4. Stretch: the most-referenced table (AdventureWorks)

This answer runs on SQL Server, so set up AdventureWorks first. In Colab that takes a few minutes the first time, and on your own machine it only checks the connection.

In [14]:
# Run this once before any %%tsql cell.
# In Google Colab it installs SQL Server and AdventureWorks into this session (about 3 to 5
# minutes the first time, and again in each new session, because Colab starts fresh).
# On your own machine it just checks that your SQL Server is reachable.
%setup_adventureworks

AdventureWorks2022 is ready, so every %%tsql cell in this notebook will run.


In [15]:
%%tsql
SELECT TOP 5
    OBJECT_SCHEMA_NAME(referenced_object_id) + '.' + OBJECT_NAME(referenced_object_id) AS referenced_table,
    COUNT(*) AS foreign_keys
FROM sys.foreign_keys
GROUP BY
    referenced_object_id
ORDER BY
    foreign_keys DESC,
    referenced_table;

referenced_table,foreign_keys
Production.Product,14
Person.Person,7
HumanResources.Employee,6
Person.BusinessEntity,5
Sales.SalesTerritory,5


`Production.Product`, with 14 foreign keys pointing at it: inventory, price history, reviews, purchase orders, work orders, and the bill of materials, which points at it twice (once for the assembly, once for the component). Those 14 rules are why a product can't be deleted while anything still refers to it, and the system views are how you find out what refers to it before you try.

---

[← Back to 08b - Changing Data, Part 2](<../08b-changing-data-part-2.ipynb>) · [Course Overview](<../00-course-overview.ipynb>)

*SQL_Tutorial* is written and maintained by Samuel Shaibu as part of *All About Data & More*. Licensed under [MIT](../LICENSE).